### 해양 모델(RTOFS/HYCOM 등) 격자 데이터를 일반적인 지리적 위도/경도 에 일치시키기 위한 작업

* 해양 수치 모델(RTOFS, HYCOM 등)의 데이터는 일반적인 정형 격자(Rectilinear Grid)가 아니라 왜곡된 2차원 곡선 좌표계(Curvilinear Grid)를 사용합니다. 따라서 일반적인 위경도 방식처럼 1차원 좌표로 바로 잘라내거나(ds.sel()) 매핑하려고 하면 에러가 나거나 데이터가 꼬이게 됩니다.
* 일반적인 지리적 위/경도 체계에 맞춰 원활하게 데이터를 다루기 위해 xarray로 로드한 직후에 거쳐야 할 핵심 선작업(Prerequisite Steps)들을 정리해 드립니다.

In [13]:
import xarray as xr

1. 파일 오픈
  ```python
ds = xr.open_dataset('rtofs_glo_file.nc')

In [14]:
file_path="./test/rtofs_glo_uv_20261103_hcasts.nc"
ds = xr.open_dataset(file_path)

2. 경도(Longitude) 범위 정규화 (Modulo 연산)
* 이유: 모델 내부 계산 편의상 경도가 360°를 넘어 1000° 이상으로 누적되어 있는 경우가 많습니다.
* 해결 방법: 전지구 표준 범위(0° ~ 360° 또는 -180° ~ 180°)에 맞게 정규화해 줍니다.

In [3]:
ds['Longitude'] = xr.where(
    ds['Longitude'] > 360, ds['Longitude'] % 360, ds['Longitude']
)

3. 위도/Y축 방향성(정렬 상태) 확인 및 반전

* **이유:** 모델에 따라 Y축 인덱스(`0`번)가 남극에서 시작하는지, 북극에서 시작하는지가 다릅니다. 이로 인해 남북 반구가 뒤집혀 보일 수 있습니다.
* **해결 방법:** 위도 데이터의 정렬 방향이 지리적 기준과 반대라면, Y축 배열을 뒤집어(`[::-1]`) 방향을 일치시켜 줍니다.
  ```python
  # 만약 위도가 북에서 남으로 정방향이 아니라면 Y축 기준으로 반전 필요 시 적용
  # lat_2d = ds['Latitude'].values[::-1, :]

In [13]:
lat_2d = ds['Latitude'].values[::-1, :]

4. 1D .sel() 대신 2D 불리언 마스크(Boolean Mask) 활용
* 이유: 곡선 좌표계에서는 위도와 경도가 각각 독립된 1차원 축이 아니라 (Y, X) 형태의 2차원 매트릭스로 얽혀 있습니다.
따라서 ds.sel(lat=..., lon=...) 같은 1차원 선택 기능을 쓸 수 없습니다.
* 해결 방법: 사용 중이신 스크립트처럼 2차원 위경도 조건에 맞는 불리언 마스크를 만들어 영역을 필터링해야 합니다.

  ```python
# 아시아-태평양 등 특정 영역 필터링을 위한 2D 마스크 생성 예시
lat_2d = ds['Latitude'].values[::-1, :]
lon_2d = ds['Longitude'].values

# 원하는 위/경도 범위 설정
lat_min, lat_max = -10.0, 60.0
lon_min, lon_max = 90.0, 170.0

mask = (
    (lat_2d >= lat_min)
    & (lat_2d <= lat_max)
    & (lon_2d >= lon_min)
    & (lon_2d <= lon_max)
)


### 💡 요약 프로세스 코드 예시
파일을 연 직후 아래와 같은 순서로 전처리를 수행하시면 모델 고유의 왜곡을 방지하고 일반 지리 좌표계와 원활하게 연동할 수 있습니다.

```python
import xarray as xr

# 1. 파일 오픈
ds = xr.open_dataset('rtofs_glo_file.nc')

# 2. 경도 360도 초과 값 정규화
ds['Longitude'] = xr.where(
    ds['Longitude'] > 360, ds['Longitude'] % 360, ds['Longitude']
)

# 3. 위도/Y축 방향성(정렬 상태) 확인 및 반전
# 만약 위도가 북에서 남으로 정방향이 아니라면 Y축 기준으로 반전 필요 시 적용
 lat_2d = ds['Latitude'].values[::-1, :]

# 4. 이후 2D 위경도 값을 이용한 공간 슬라이싱/마스킹 진행

In [4]:
ds['Longitude'].values[0,0],  ds['Longitude'].values[-1,-1], ds['Longitude'].values.min(), ds['Longitude'].values.max()

(np.float32(74.160034),
 np.float32(74.119995),
 np.float32(0.00012207031),
 np.float32(360.0))

In [35]:
ds['Latitude'].values[0,0],  ds['Latitude'].values[-1,-1], ds['Latitude'].values.min(), ds['Latitude'].values.max()

(np.float32(-78.64),
 np.float32(47.041878),
 np.float32(-78.64),
 np.float32(89.97772))

In [5]:
ds['Longitude'] = xr.where(
    ds['Longitude'] > 360, ds['Longitude'] % 360, ds['Longitude']
)

In [6]:
ds['Latitude'].values = ds['Latitude'].values[::-1, :]
ds['u_velocity'].values = ds['u_velocity'].values[::-1, :]
ds['v_velocity'].values = ds['v_velocity'].values[::-1, :]

In [7]:
ds['Latitude'].values[0,0],  ds['Latitude'].values[-1,-1], ds['Latitude'].values.min(), ds['Latitude'].values.max()

(np.float32(47.041878),
 np.float32(-78.64),
 np.float32(-78.64),
 np.float32(89.97772))

In [8]:
ds['Longitude'].values[0,0],  ds['Longitude'].values[-1,-1], ds['Longitude'].values.min(), ds['Longitude'].values.max()

(np.float32(74.160034),
 np.float32(74.119995),
 np.float32(0.00012207031),
 np.float32(360.0))

In [9]:
lon_2d = ds['Longitude'].values
lat_2d = ds['Latitude'].values

In [10]:
mask = (
(lat_2d >= -10.0)
& (lat_2d <= 60.0)
& (lon_2d >= 90.0)
& (lon_2d <= 180.0)
)

In [11]:
import numpy as np

y_indices, x_indices = np.where(mask)
ymin, ymax = y_indices.min(), y_indices.max()
xmin, xmax = x_indices.min(), x_indices.max()
print(f"추출된 격자 인덱스 범위 -> Y: {ymin}~{ymax}, X: {xmin}~{xmax}")


추출된 격자 인덱스 범위 -> Y: 0~1918, X: 198~1323


In [12]:
u_sub = ds['u_velocity'].isel(MT=0, Layer=0, Y=slice(ymin, ymax + 1), X=slice(xmin, xmax + 1)).values
v_sub = ds['v_velocity'].isel(MT=0, Layer=0, Y=slice(ymin, ymax + 1), X=slice(xmin, xmax + 1)).values

print(f"슬라이싱된 범위 -> u_sub: {u_sub.shape}, v_sub: {v_sub.shape}")


슬라이싱된 범위 -> u_sub: (1919, 1126), v_sub: (1919, 1126)


In [13]:
lat_sub = lat_2d[ymin:ymax+1, xmin:xmax+1]
lon_sub = lon_2d[ymin:ymax+1, xmin:xmax+1]

print(f"슬라이싱된 범위 ->  lon_sub: {lon_sub.shape}, lat_sub: {lat_sub.shape}")

슬라이싱된 범위 ->  lon_sub: (1919, 1126), lat_sub: (1919, 1126)


In [20]:
lo1 = round(float(lon_sub[0, 0]), 5)
la1 = round(float(lat_sub[0, 0]), 5)
lo2 = round(float(lon_sub[-1, -1]), 5)
la2 = round(float(lat_sub[-1, -1]), 5)

In [63]:
#"nx":1080,"ny":481,"lo1":20,"la1":80,"lo2":379.6666666666667,"la2":-80,"dx":0.3333333333333333,"dy":0.333

In [21]:
ny, nx = u_sub.shape

In [22]:
u_flat = [None if np.isnan(val) else round(float(val), 2) for val in u_sub.flatten()]
v_flat = [None if np.isnan(val) else round(float(val), 2) for val in v_sub.flatten()]


In [6]:
import pandas as pd

In [15]:


idx_array = ds['MT'].values
#chk_time = pd.to_datetime(idx_array[0]).strftime('%Y-%m-%dT%H:%M:%S.000Z')


In [16]:
aa = str(idx_array[0])

In [5]:
mt_idx = 8
forcastTime = int(mt_idx * 3)